# R24 FANOUT free-gate tier (b) - six CPU adjudications

**Author**: Claude (opus executor)
**Date**: 2026-07-08
**Approach**: pure-CPU resampling of the frozen H119 checkpoints (arm `A_production`, 5 runs x 10 docs), the H248 lexical scanner, the H119 chunk cache and the h195 probe catalogue. Zero LLM / GPU / Neo4j. Adjudicates six registered R24 fan-out gates: H255 (adaptive-K allocation), H256 (completeness instruments), H258 (internal-dedup proxy), H259 (salience shadowing), H261 (leakage matrix) and H263 (class concentration).

Each gate is scored against its **registered bar** and returns a PASS/FAIL with a `verdict_recommendation` and explicit caveats. Anchors reproduced first: union-of-5 = 63 gold carriers, single-run mean 76.8% of union-5.

In [1]:
# Imports - grouped by category
import os                                        # cwd normalization under nbconvert
import json                                      # artifact + report serialization
import glob                                      # checkpoint discovery
import pickle                                    # chunk cache
import re                                        # H248 scanner + tokenizer
import random                                    # random-allocation control (H255b)
import itertools                                 # run-subset enumeration
from collections import Counter                  # incidence counting
from statistics import mean, median, pstdev      # metric aggregation
from datetime import datetime, timezone          # UTC report stamp
from pathlib import Path

from rapidfuzz import fuzz                        # frozen token_set_ratio matcher
from scipy.stats import spearmanr                # H256(b) rank correlation
from knowledge_graph_foundry.models import normalize_name
from knowledge_graph_foundry.config import PROJ_ROOT
from rich.console import Console
from rich.table import Table
from rich import box

os.chdir(PROJ_ROOT)
console = Console()
print("imports ok; cwd:", os.getcwd())

2026-07-08 08:55:18.007 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


imports ok; cwd: /home/lab/workspace/learning/projects/knowledge-graph-foundry


## Configuration and frozen harness

Reconstructs the R23 harness exactly: checkpoint loader (`arm -> run -> doc -> [names]`), frozen `token_set_ratio >= 85` gold-carrier matcher against the 101 probe products, the concatenated chunk text per doc, and the H248 three-pattern lexical scanner. Derives the shared per-doc structures every gate consumes.

In [2]:
# --- Frozen config ---
THR = 85
CKPT_GLOB = "results/h119/*.json"
CHUNK_CACHE = Path("data/interim/h119_chunks.pkl")
PROBES = Path("data/processed/probes-wide-v2-h195.json")
REPORTS_DIR = Path("reports"); REPORTS_DIR.mkdir(exist_ok=True)
LOG_PATH = Path("logs/r24b-fanout-gates.log")

def log(msg):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    with open(LOG_PATH, "a") as fh:
        fh.write(f"[{stamp}] {msg}\n")
log("=== r24b fanout gates start ===")

def norm(name):
    base = normalize_name(name)
    return " ".join(base.replace("-", " ").split())

# --- checkpoints ---
ARMS = {}
for f in glob.glob(CKPT_GLOB):
    d = json.load(open(f))
    ARMS.setdefault(d["arm"], {}).setdefault(d["run"], {})[d["doc"]] = d["names"]
A = ARMS["A_production"]
RUNS = sorted(A); DOCS = sorted(A[1])

# --- gold carriers ---
PRODUCTS = list(dict.fromkeys(p["product"] for p in json.load(open(PROBES))["probes"]))
PLOW = [p.lower() for p in PRODUCTS]

def matched(ent_list):
    el = [e.lower() for e in ent_list]
    return {i for i, pl in enumerate(PLOW)
            if any(fuzz.token_set_ratio(e, pl) >= THR for e in el)}

def run_ents(arm, r):
    return [n for doc in DOCS for n in arm[r].get(doc, [])]

perA = {r: matched(run_ents(A, r)) for r in RUNS}
UNION5 = set().union(*perA.values()); U = len(UNION5)
single_cov = mean(len(perA[r]) / U for r in RUNS)

# --- chunks ---
chunks = pickle.load(open(CHUNK_CACHE, "rb"))
DOCTEXT = {doc: "".join(c["text"] for c in sorted(chunks[doc], key=lambda c: c["index"]))
           for doc in DOCS}
DOCTOKENS = {doc: sum(c["token_count"] for c in chunks[doc]) for doc in DOCS}

# --- per-doc structures ---
docsets = {doc: {r: set(norm(x) for x in A[r].get(doc, [])) for r in RUNS} for doc in DOCS}
perAdoc = {doc: {r: matched(A[r].get(doc, [])) for r in RUNS} for doc in DOCS}
UNION5_doc = {doc: set().union(*perAdoc[doc].values()) for doc in DOCS}
obs_union_ent = {doc: len(set().union(*docsets[doc].values())) for doc in DOCS}

# --- H248 scanner (per doc) ---
CODE = re.compile(r"\b(?=[A-Za-z0-9\-]*[A-Za-z])(?=[A-Za-z0-9\-]*\d)[A-Za-z][A-Za-z0-9\-]{1,}\b")
MULTI = re.compile(r"\b[A-Z][a-zA-Z]+(?:\s+(?:[A-Z][a-zA-Z0-9]*|[A-Z0-9]{2,}|\d+[A-Za-z]*)){1,5}\b")
ALLCAPS = re.compile(r"\b[A-Z]{2,}[A-Z0-9]*\b")
SCAN_DOC = {}
for doc in DOCS:
    cand = set()
    for rx in (CODE, MULTI, ALLCAPS):
        for m in rx.finditer(DOCTEXT[doc]):
            cand.add(m.group(0).strip())
    SCAN_DOC[doc] = cand
SCAN_HIT = {}
for doc in DOCS:
    cl = [c.lower() for c in SCAN_DOC[doc]]
    SCAN_HIT[doc] = {i for i, pl in enumerate(PLOW)
                     if any(fuzz.token_set_ratio(c, pl) >= THR for c in cl)}

RESULTS = {}

# --- anchors ---
tbl = Table(box=box.SIMPLE, show_edge=False)
tbl.add_column("anchor"); tbl.add_column("value", justify="right")
tbl.add_row("arms loaded", ", ".join(sorted(ARMS)))
tbl.add_row("runs x docs", f"{len(RUNS)} x {len(DOCS)}")
tbl.add_row("probe products", str(len(PRODUCTS)))
tbl.add_row("union-of-5 carriers (U)", str(U))
tbl.add_row("single-run mean cov of U", f"{single_cov:.1%}")
tbl.add_row("sum per-doc union5 slots", str(sum(len(v) for v in UNION5_doc.values())))
console.print(tbl)
assert U == 63 and abs(single_cov - 0.768) < 0.01, "anchor drift"
print("anchors reproduce: union5=63, single-run 76.8%")
log(f"anchors U={U} single_cov={single_cov:.3f}")

 anchor                                                        value 
─────────────────────────────────────────────────────────────────────
 arms loaded                A_production, B_canonical, C_attribution 
 runs x docs                                                  5 x 10 
 probe products                                                  101 
 union-of-5 carriers (U)                                          63 
 single-run mean cov of U                                      76.8% 
 sum per-doc union5 slots                                        214

anchors reproduce: union5=63, single-run 76.8%


## H255 - Adaptive-K allocation (spend passes where the residue is)

Three clauses on arm A. **(a)** Chao2 richness from 3-pass subsets predicts the observed union-of-5 entity count within +-15% median absolute error. **(b)** at a matched mean-2-pass budget, residue-greedy allocation recovers >= 5 pts more gold carriers than uniform-2. **(c)** an adaptive stop at estimated coverage >= 0.95 reaches >= 90% of a doc's union-5 carriers with < 5 passes on >= 5 of 10 docs. Names the winning controller. Failed extractions (runs returning 0 names) are kept as honest zero-yield passes.

In [3]:
# --- Chao2 incidence estimator over a list of pass entity-sets ---
def chao2(passsets):
    inc = Counter()
    for s in passsets:
        for e in s:
            inc[e] += 1
    S = len(inc)
    if S == 0:
        return 0.0
    f1 = sum(v == 1 for v in inc.values())
    f2 = sum(v == 2 for v in inc.values())
    if f2 > 0:
        return S + f1 * f1 / (2 * f2)
    return S + f1 * (f1 - 1) / (2 * (f2 + 1))   # bias-corrected f2=0 form

# ===== H255(a): Chao2 from 3-pass subsets vs observed union-of-5 entity count =====
subsets3 = list(itertools.combinations(RUNS, 3))
errs = []; rows_a = []
for doc in DOCS:
    obs = obs_union_ent[doc]
    if obs == 0:
        continue
    est = mean(chao2([docsets[doc][r] for r in sub]) for sub in subsets3)
    e = (est - obs) / obs
    errs.append(e); rows_a.append((doc, obs, est, e))
mae = median(abs(e) for e in errs)
med_bias = median(errs)
a_pass = mae <= 0.15
print("H255(a) Chao2(3-pass) vs observed union-of-5 entity count:")
for doc, obs, est, e in sorted(rows_a, key=lambda x: x[1]):
    print(f"  {doc[:34]:34s} obs={obs:4d} chao2={est:7.1f}  err={e:+.1%}")
print(f"  median |err| = {mae:.1%}  (bar <= 15%)  median signed bias={med_bias:+.1%}  -> {'PASS' if a_pass else 'FAIL'}")

H255(a) Chao2(3-pass) vs observed union-of-5 entity count:
  Airsense-Brochure.pdf              obs=  26 chao2=  143.1  err=+450.4%
  0-20190113114505.pdf               obs=  48 chao2=  705.6  err=+1370.0%
  1017900r4_ResMed_Product_Catalogue obs=  66 chao2=  341.8  err=+417.8%
  Brochure_BMC_GIII_A20_Oxygenium_Me obs=  66 chao2=  244.9  err=+271.0%
  CPAP-V3-MKT-01-CPAP-Brochure-2.00E obs=  91 chao2=  373.2  err=+310.1%
  BC-Dreamstation-Standard-CPAP.pdf  obs=  95 chao2=  512.2  err=+439.2%
  CPAP-Machines-Brochure.pdf         obs= 132 chao2=  232.2  err=+75.9%
  BMC_RESmart_AutoCPAP_User_Manual.p obs= 315 chao2=  410.7  err=+30.4%
  3B_User-Manual_CPAP-Auto-CPAP_RESm obs= 357 chao2=  555.4  err=+55.6%
  ARTP_Standards_of_Care_-_CPAP_Devi obs= 475 chao2= 1062.0  err=+123.6%
  median |err| = 290.6%  (bar <= 15%)  median signed bias=+290.6%  -> FAIL


In [4]:
# ===== H255(b): residue-greedy allocation vs uniform-2 / random, matched budget =====
def gt_residue(doc, m):        # Good-Turing f1/k new-entity rate over first m passes
    inc = Counter()
    for r in RUNS[:m]:
        for e in docsets[doc][r]:
            inc[e] += 1
    f1 = sum(v == 1 for v in inc.values())
    return f1 / m
def chao2_residue(doc, m):     # estimated unseen mass over first m passes
    ps = [docsets[doc][r] for r in RUNS[:m]]
    S = len(set().union(*ps)) if ps else 0
    return max(chao2(ps) - S, 0.0)
def scanner_residue(doc, m):   # scanner carriers still unextracted after m passes
    got = set().union(*[perAdoc[doc][r] for r in RUNS[:m]]) if m else set()
    return len(SCAN_HIT[doc] - got)

def greedy(score_fn, extra=10):
    passes = {d: 1 for d in DOCS}
    for _ in range(extra):
        cand = [d for d in DOCS if passes[d] < 5]
        if not cand:
            break
        best = max(cand, key=lambda d: score_fn(d, passes[d]))
        passes[best] += 1
    return passes

def recovered(passes):
    got = set()
    for d in DOCS:
        for r in RUNS[:passes[d]]:
            got |= perAdoc[d][r]
    return len(got & UNION5)

ctrls = {"chao2": chao2_residue, "good_turing": gt_residue, "scanner": scanner_residue}
uni2 = recovered({d: 2 for d in DOCS})
rnd = []
for seed in range(20):
    rng = random.Random(seed); passes = {d: 1 for d in DOCS}
    for _ in range(10):
        cand = [d for d in DOCS if passes[d] < 5]
        passes[rng.choice(cand)] += 1
    rnd.append(recovered(passes))
rnd_mean = mean(rnd)

def pts(n): return 100 * n / U
b_rows = {name: recovered(greedy(fn)) for name, fn in ctrls.items()}
best_ctrl = max(b_rows, key=b_rows.get)
best_rec = b_rows[best_ctrl]
b_gap = pts(best_rec) - pts(uni2)
b_pass = b_gap >= 5.0
print("H255(b) matched-budget (mean 2 passes/doc) gold-carrier recovery:")
print(f"  uniform-2 : {uni2}/{U} = {pts(uni2):.1f}%")
print(f"  random    : {rnd_mean:.1f}/{U} = {pts(rnd_mean):.1f}%  (20 seeds)")
for name in ctrls:
    print(f"  greedy[{name:11s}]: {b_rows[name]}/{U} = {pts(b_rows[name]):.1f}%")
print(f"  winner={best_ctrl}  greedy-uniform = {b_gap:+.1f} pts  (bar >= +5)  -> {'PASS' if b_pass else 'FAIL'}")

H255(b) matched-budget (mean 2 passes/doc) gold-carrier recovery:
  uniform-2 : 60/63 = 95.2%
  random    : 57.9/63 = 91.9%  (20 seeds)
  greedy[chao2      ]: 55/63 = 87.3%
  greedy[good_turing]: 55/63 = 87.3%
  greedy[scanner    ]: 62/63 = 98.4%
  winner=scanner  greedy-uniform = +3.2 pts  (bar >= +5)  -> FAIL


In [5]:
# ===== H255(c): adaptive stop at estimated coverage >= 0.95 =====
def gt_coverage(doc, m):
    inc = Counter()
    for r in RUNS[:m]:
        for e in docsets[doc][r]:
            inc[e] += 1
    n = sum(inc.values())
    if n == 0:
        return 0.0
    f1 = sum(v == 1 for v in inc.values())
    return 1 - f1 / n

c_rows = []; n_good = 0
for doc in DOCS:
    if not UNION5_doc[doc]:
        continue
    m = 1
    while m < 5 and gt_coverage(doc, m) < 0.95:
        m += 1
    got = set().union(*[perAdoc[doc][r] for r in RUNS[:m]])
    recall = len(got & UNION5_doc[doc]) / len(UNION5_doc[doc])
    good = (m < 5) and (recall >= 0.90)
    n_good += good
    c_rows.append((doc, m, gt_coverage(doc, m), recall, good))
c_pass = n_good >= 5
print("H255(c) adaptive-stop (GT coverage >= 0.95):")
for doc, m, cov, rec, g in c_rows:
    print(f"  {doc[:34]:34s} passes={m}  cov={cov:.2f}  union5-recall={rec:.0%}  {'OK' if g else '-'}")
print(f"  docs with <5 passes AND >=90% union5-recall: {n_good}/10  (bar >= 5)  -> {'PASS' if c_pass else 'FAIL'}")

h255_verdict = ("CONFIRMED" if (a_pass and b_pass and c_pass)
                else "PARTIAL" if (a_pass or b_pass or c_pass) else "REFUTED")
RESULTS["H255"] = {
    "clause_a_chao2_mae": {"median_abs_err": round(mae, 4), "median_signed_bias": round(med_bias, 4),
                           "bar": 0.15, "pass": bool(a_pass)},
    "clause_b_greedy": {"winning_controller": best_ctrl, "greedy_recovered": best_rec,
                        "uniform2_recovered": uni2, "random_recovered": round(rnd_mean, 1),
                        "gap_pts": round(b_gap, 2), "bar_pts": 5, "pass": bool(b_pass),
                        "all_controllers": dict(b_rows)},
    "clause_c_adaptive_stop": {"docs_meeting": n_good, "bar_docs": 5, "pass": bool(c_pass)},
    "winning_controller": best_ctrl,
    "verdict_recommendation": h255_verdict,
    "caveat": "failed extractions (0-name runs) kept as zero-yield passes; incidence counts reflect real pipeline flakiness, which depresses Chao2 richness and inflates single-pass residue",
}
print("H255 verdict:", h255_verdict)
log(f"H255 a_mae={mae:.3f} b_gap={b_gap:.1f}({best_ctrl}) c_good={n_good} -> {h255_verdict}")

H255(c) adaptive-stop (GT coverage >= 0.95):
  0-20190113114505.pdf               passes=5  cov=0.00  union5-recall=100%  -
  1017900r4_ResMed_Product_Catalogue passes=5  cov=0.78  union5-recall=100%  -
  3B_User-Manual_CPAP-Auto-CPAP_RESm passes=5  cov=0.67  union5-recall=100%  -
  ARTP_Standards_of_Care_-_CPAP_Devi passes=5  cov=0.51  union5-recall=100%  -
  Airsense-Brochure.pdf              passes=5  cov=0.73  union5-recall=100%  -
  BC-Dreamstation-Standard-CPAP.pdf  passes=5  cov=0.69  union5-recall=100%  -
  BMC_RESmart_AutoCPAP_User_Manual.p passes=5  cov=0.71  union5-recall=100%  -
  Brochure_BMC_GIII_A20_Oxygenium_Me passes=5  cov=0.52  union5-recall=100%  -
  CPAP-Machines-Brochure.pdf         passes=5  cov=0.66  union5-recall=100%  -
  CPAP-V3-MKT-01-CPAP-Brochure-2.00E passes=5  cov=0.46  union5-recall=100%  -
  docs with <5 passes AND >=90% union5-recall: 0/10  (bar >= 5)  -> FAIL
H255 verdict: REFUTED


## H256 - Completeness instruments (missing mass from one pass)

**(a)** Chapman two-source estimator per (doc, run) with n1 = run entities, n2 = scanner candidates, m = overlap; compared to the observed union-of-5 entity count - within +-25% or a stable measurable bias. Also quantifies the co-miss rate (gold carriers missed by BOTH the single run and the scanner). **(b)** per-doc Good-Turing coverage from 2-pass subsets rank-correlates with those subsets' actual gold-carrier recall at Spearman rho >= 0.6.

In [6]:
# ===== H256(a): Chapman estimator per (doc, run) =====
biases = []; co_miss = 0; co_tot = 0
for doc in DOCS:
    cl = [c.lower() for c in SCAN_DOC[doc]]
    n2 = len(SCAN_DOC[doc])
    obs = obs_union_ent[doc]
    for r in RUNS:
        ents = [e.lower() for e in A[r].get(doc, [])]
        n1 = len(docsets[doc][r])
        if n1 == 0 or obs == 0 or n2 == 0:
            continue
        m = sum(1 for c in cl if any(fuzz.token_set_ratio(c, e) >= THR for e in ents))
        n_hat = (n1 + 1) * (n2 + 1) / (m + 1) - 1
        biases.append((n_hat - obs) / obs)
    for r in RUNS:
        for idx in UNION5_doc[doc]:
            co_tot += 1
            if idx not in perAdoc[doc][r] and idx not in SCAN_HIT[doc]:
                co_miss += 1
mean_bias = mean(biases); std_bias = pstdev(biases)
co_rate = co_miss / co_tot if co_tot else 0.0
a2_within = abs(mean_bias) <= 0.25
a2_stable = std_bias < abs(mean_bias)          # bias dominates its own scatter -> "stable measurable bias"
a2_pass = a2_within or a2_stable
print("H256(a) Chapman (LLM run x scanner) vs observed union-of-5 entity count:")
print(f"  n={len(biases)} (doc,run) cells;  mean relative bias={mean_bias:+.1%}  std={std_bias:.1%}")
print(f"  within +-25%: {a2_within}   stable-measurable-bias (std<|mean|): {a2_stable}  -> {'PASS' if a2_pass else 'FAIL'}")
print(f"  co-miss rate (carrier missed by BOTH run and scanner): {co_rate:.1%}  ({co_miss}/{co_tot} slots)")

H256(a) Chapman (LLM run x scanner) vs observed union-of-5 entity count:
  n=36 (doc,run) cells;  mean relative bias=+47.8%  std=55.2%
  within +-25%: False   stable-measurable-bias (std<|mean|): False  -> FAIL
  co-miss rate (carrier missed by BOTH run and scanner): 26.7%  (286/1070 slots)


In [7]:
# ===== H256(b): GT coverage (2-pass) vs actual gold-carrier recall, Spearman =====
pairs2 = list(itertools.combinations(RUNS, 2))
cov_x, rec_y = [], []
for doc in DOCS:
    if not UNION5_doc[doc]:
        continue
    denom = len(UNION5_doc[doc])
    for a_, b_ in pairs2:
        inc = Counter()
        for r in (a_, b_):
            for e in docsets[doc][r]:
                inc[e] += 1
        n = sum(inc.values())
        if n == 0:
            continue
        f1 = sum(v == 1 for v in inc.values())
        C = 1 - f1 / n
        got = perAdoc[doc][a_] | perAdoc[doc][b_]
        recall = len(got & UNION5_doc[doc]) / denom
        cov_x.append(C); rec_y.append(recall)
rho, pval = spearmanr(cov_x, rec_y)
b2_pass = rho >= 0.6
print("H256(b) GT coverage (2-pass) vs actual union5 recall:")
print(f"  n={len(cov_x)} (doc,pair) cells;  Spearman rho={rho:.3f}  p={pval:.1e}  (bar >= 0.6)  -> {'PASS' if b2_pass else 'FAIL'}")

h256_verdict = "CONFIRMED" if (a2_pass and b2_pass) else "PARTIAL" if (a2_pass or b2_pass) else "REFUTED"
RESULTS["H256"] = {
    "clause_a_chapman": {"mean_rel_bias": round(mean_bias, 4), "std_rel_bias": round(std_bias, 4),
                         "within_25pct": bool(a2_within), "stable_bias": bool(a2_stable),
                         "co_miss_rate": round(co_rate, 4), "pass": bool(a2_pass)},
    "clause_b_gt_coverage": {"spearman_rho": round(float(rho), 4), "p": float(pval),
                             "n_cells": len(cov_x), "bar": 0.6, "pass": bool(b2_pass)},
    "verdict_recommendation": h256_verdict,
    "caveat": "scanner over-generates surface forms so n2 is large and Chapman biases positive (not the co-miss negative bias the grounding anticipated); co-miss rate reported as the honest lower-bound signal",
}
print("H256 verdict:", h256_verdict)
log(f"H256 chapman_bias={mean_bias:.2f}+-{std_bias:.2f} co_miss={co_rate:.2f} rho={rho:.2f} -> {h256_verdict}")

H256(b) GT coverage (2-pass) vs actual union5 recall:
  n=88 (doc,pair) cells;  Spearman rho=0.095  p=3.8e-01  (bar >= 0.6)  -> FAIL
H256 verdict: REFUTED


## H258 - Internal-dedup signal (granularity-adapted)

The registered within-document form (a carrier missed at one position but emitted elsewhere in the SAME document by the same run) is **untestable at this granularity**: the H119 checkpoints are doc-level entity sets with no chunk-level or positional attribution, so "elsewhere in the same document" collapses to the doc set itself. Adapted honestly to the registered fallback - the **cross-document proxy**: of each run's gold carriers missed in doc d, how many did the SAME run extract in one of its OTHER doc checkpoints (known-but-locally-dropped). Bar analog >= 30%.

In [8]:
# confirm no chunk-level attribution exists in the checkpoint schema
sample = json.load(open(glob.glob(CKPT_GLOB)[0]))
print("checkpoint keys:", list(sample.keys()), "-> names are doc-level, no per-chunk provenance")

# ===== H258 cross-document proxy =====
missed_tot = 0; found_elsewhere = 0
for r in RUNS:
    for d in DOCS:
        missed = UNION5_doc[d] - perAdoc[d][r]
        for idx in missed:
            missed_tot += 1
            if any(idx in perAdoc[d2][r] for d2 in DOCS if d2 != d):
                found_elsewhere += 1
rate = found_elsewhere / missed_tot if missed_tot else 0.0
h258_pass = rate >= 0.30
print(f"H258 cross-doc proxy: {found_elsewhere}/{missed_tot} = {rate:.1%} of a run's locally-missed carriers were extracted by the SAME run in another doc  (bar >= 30%)  -> {'PASS' if h258_pass else 'FAIL'}")

h258_verdict = "CONFIRMED (cross-doc proxy)" if h258_pass else "REFUTED (misses genuinely unseen)"
RESULTS["H258"] = {
    "granularity_caveat": "checkpoints are doc-level entity sets; the registered within-document dedup signal is untestable here - no chunk/positional provenance. Cross-document proxy substituted per the registered fallback.",
    "cross_doc_proxy_rate": round(rate, 4), "n_missed_slots": missed_tot,
    "n_found_elsewhere": found_elsewhere, "bar": 0.30, "pass": bool(h258_pass),
    "verdict_recommendation": h258_verdict,
}
print("H258 verdict:", h258_verdict)
log(f"H258 cross_doc_proxy={rate:.3f} ({found_elsewhere}/{missed_tot}) -> {h258_verdict}")

checkpoint keys: ['arm', 'run', 'doc', 'names', 'count', 'seconds'] -> names are doc-level, no per-chunk provenance
H258 cross-doc proxy: 432/531 = 81.4% of a run's locally-missed carriers were extracted by the SAME run in another doc  (bar >= 30%)  -> PASS
H258 verdict: CONFIRMED (cross-doc proxy)


## H259 - Salience shadowing

For each (doc, run), locate each union-5 gold carrier's mention in the chunk text (substring / token match) and compute that chunk's **entity density** = extracted run-entities located in the chunk per 1000 chunk tokens. Compare the density of chunks around MISSED carriers vs CAPTURED carriers. Bar: missed-carrier chunk density >= 1.5x captured. Caveat: five of ten docs are single-chunk, where every carrier shares one density value and the within-doc contrast vanishes.

In [9]:
def locate_chunk(doc, needle):
    nl = needle.lower()
    cs = sorted(chunks[doc], key=lambda c: c["index"])
    for c in cs:
        if nl in c["text"].lower():
            return c["index"]
    toks = set(nl.split())
    best, bestscore = None, 0
    for c in cs:
        ctl = c["text"].lower()
        sc = sum(t in ctl for t in toks)
        if sc > bestscore:
            bestscore, best = sc, c["index"]
    return best if bestscore >= max(1, len(toks) - 1) else None

chunk_by_index = {doc: {c["index"]: c for c in chunks[doc]} for doc in DOCS}
def chunk_density(doc, r, cidx):
    c = chunk_by_index[doc][cidx]
    ctl = c["text"].lower()
    n = sum(1 for e in docsets[doc][r] if e and e in ctl)
    return 1000.0 * n / c["token_count"]

missed_d, capt_d = [], []
single_chunk_docs = [d for d in DOCS if len(chunks[d]) == 1]
for doc in DOCS:
    for idx in UNION5_doc[doc]:
        cidx = locate_chunk(doc, PRODUCTS[idx])
        if cidx is None:
            continue
        for r in RUNS:
            if not docsets[doc][r]:
                continue
            dens = chunk_density(doc, r, cidx)
            (capt_d if idx in perAdoc[doc][r] else missed_d).append(dens)
mm, cm = mean(missed_d), mean(capt_d)
ratio = mm / cm if cm else float("inf")
h259_pass = ratio >= 1.5
print("H259 salience shadowing (entity density per 1000 tokens of the mention chunk):")
print(f"  missed   carriers: mean density={mm:.2f}  (n={len(missed_d)})")
print(f"  captured carriers: mean density={cm:.2f}  (n={len(capt_d)})")
print(f"  missed / captured = {ratio:.2f}x  (bar >= 1.5x)  -> {'PASS' if h259_pass else 'FAIL'}")
print(f"  caveat: {len(single_chunk_docs)}/10 docs are single-chunk (density identical for all their carriers)")

h259_verdict = "CONFIRMED" if h259_pass else "REFUTED (shadowing signal absent)"
RESULTS["H259"] = {
    "missed_mean_density": round(mm, 3), "captured_mean_density": round(cm, 3),
    "ratio": round(ratio, 3), "bar": 1.5,
    "n_missed": len(missed_d), "n_captured": len(capt_d),
    "single_chunk_docs": len(single_chunk_docs), "pass": bool(h259_pass),
    "verdict_recommendation": h259_verdict,
    "caveat": "mention located by substring/token match; single-chunk docs (4/10) give an identical density to missed and captured carriers, muting the within-doc contrast",
}
print("H259 verdict:", h259_verdict)
log(f"H259 missed={mm:.2f} capt={cm:.2f} ratio={ratio:.2f} -> {h259_verdict}")

H259 salience shadowing (entity density per 1000 tokens of the mention chunk):
  missed   carriers: mean density=10.10  (n=138)
  captured carriers: mean density=13.07  (n=255)
  missed / captured = 0.77x  (bar >= 1.5x)  -> FAIL
  caveat: 5/10 docs are single-chunk (density identical for all their carriers)
H259 verdict: REFUTED (shadowing signal absent)


## H261 - Leakage matrix (SELECTS the clean demo split)

Pairwise lexical overlap between the 10 docs: token Jaccard on chunk-text vocabularies plus shared gold-carrier count. No bar - this gate selects the low-leakage demo/target design for the downstream in-context distillation LLM probe (H261 full test). Reports the three cleanest demo-doc candidates (lowest mean Jaccard).

In [10]:
WORD = re.compile(r"[a-z0-9]{2,}")
vocab = {doc: set(WORD.findall(DOCTEXT[doc].lower())) for doc in DOCS}
def jac(a, b):
    u = vocab[a] | vocab[b]
    return len(vocab[a] & vocab[b]) / len(u) if u else 0.0

pairs = []
for i, j in itertools.combinations(DOCS, 2):
    pairs.append((i, j, jac(i, j), len(UNION5_doc[i] & UNION5_doc[j])))
mean_j = {d: mean(jac(d, o) for o in DOCS if o != d) for d in DOCS}
cleanest = sorted(DOCS, key=lambda d: mean_j[d])[:3]

print("H261 leakage - mean token-Jaccard of each doc to the other nine:")
for d in sorted(DOCS, key=lambda d: mean_j[d]):
    print(f"  {mean_j[d]:.3f}  {d[:44]}")
print("\n  3 cleanest demo-doc candidates (lowest leakage):")
for d in cleanest:
    print(f"    - {d}  (mean J={mean_j[d]:.3f})")
print("\n  cleanest doc-pairs (fewest shared carriers, low Jaccard):")
for i, j, jj, sh in sorted(pairs, key=lambda x: (x[3], x[2]))[:5]:
    print(f"    J={jj:.3f}  shared_carriers={sh}  {i[:24]} | {j[:24]}")

RESULTS["H261"] = {
    "no_bar": True,
    "mean_jaccard_per_doc": {d: round(mean_j[d], 4) for d in DOCS},
    "cleanest_demo_docs": cleanest,
    "cleanest_pairs": [{"a": i, "b": j, "jaccard": round(jj, 4), "shared_carriers": sh}
                       for i, j, jj, sh in sorted(pairs, key=lambda x: (x[3], x[2]))[:5]],
    "verdict_recommendation": "SELECTION: use the 3 low-leakage docs as held-out demos for the H261 LLM distillation probe",
}
log(f"H261 cleanest_demos={cleanest}")

H261 leakage - mean token-Jaccard of each doc to the other nine:
  0.087  CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf
  0.118  ARTP_Standards_of_Care_-_CPAP_Devices_(Techn
  0.120  CPAP-Machines-Brochure.pdf
  0.126  Brochure_BMC_GIII_A20_Oxygenium_Medical.pdf
  0.127  BC-Dreamstation-Standard-CPAP.pdf
  0.128  Airsense-Brochure.pdf
  0.144  0-20190113114505.pdf
  0.149  1017900r4_ResMed_Product_Catalogue_ANZ_Eng_L
  0.194  BMC_RESmart_AutoCPAP_User_Manual.pdf
  0.200  3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1

  3 cleanest demo-doc candidates (lowest leakage):
    - CPAP-V3-MKT-01-CPAP-Brochure-2.00EN_NEW.pdf  (mean J=0.087)
    - ARTP_Standards_of_Care_-_CPAP_Devices_(Technical_and_Performance)_Version_5.0_-_05-02-2022.pdf  (mean J=0.118)
    - CPAP-Machines-Brochure.pdf  (mean J=0.120)

  cleanest doc-pairs (fewest shared carriers, low Jaccard):
    J=0.075  shared_carriers=0  Airsense-Brochure.pdf | CPAP-V3-MKT-01-CPAP-Broc
    J=0.091  shared_carriers=0  ARTP_Standards_of_Care_

## H263 - Class concentration

Maps the 101 probed carriers (and the 63 coverable) to entity classes using the H230 name-shape classifier (code-bearing / product-form / generic-concept / fragment) - the checkpoints carry no types in discovery mode. The probe catalogue's `attribute` / `derivation_rule` fields classify probes, not carriers, so the name-shape classifier is the usable carrier-level mapping. Bar: carriers concentrate in <= 40% of classes (fewest classes covering >= 90% of carriers, as a fraction of the 4-class space).

In [11]:
CODE_TOKEN = re.compile(r"\b(?=\w*[A-Za-z])(?=\w*\d)\w+\b")
TRADEMARK = re.compile(r"[™®©]")
def rnorm(s):
    return " ".join(TRADEMARK.sub("", s).lower().replace("-", " ").split())
multitoken_tokens = set()
for p in PRODUCTS:
    t = rnorm(p).split()
    if len(t) >= 2:
        multitoken_tokens.update(t)
def shape_of(name):
    n = rnorm(name)
    if CODE_TOKEN.search(n):
        return "code-bearing"
    if len(n.split()) >= 2:
        return "product-form"
    return "fragment" if n in multitoken_tokens else "generic-concept"

CLASSES = ["code-bearing", "product-form", "generic-concept", "fragment"]
def concentration(idxset, label):
    comp = Counter(shape_of(PRODUCTS[i]) for i in idxset)
    tot = sum(comp.values())
    cum = 0; k = 0
    for cls, n in comp.most_common():
        cum += n; k += 1
        if cum / tot >= 0.90:
            break
    frac = k / len(CLASSES)
    print(f"H263 {label} (n={tot}):")
    for cls in CLASSES:
        print(f"    {cls:16s} {comp[cls]:4d}  {100*comp[cls]/tot:5.1f}%")
    print(f"    classes covering >=90% = {k}/{len(CLASSES)} = {frac:.0%}  (bar <= 40%)  -> {'PASS' if frac <= 0.40 else 'FAIL'}")
    return dict(comp), k, frac

comp101, k101, f101 = concentration(range(len(PRODUCTS)), "101 probed carriers")
comp63, k63, f63 = concentration(sorted(UNION5), "63 coverable carriers")

drules = Counter(p["derivation_rule"] for p in json.load(open(PROBES))["probes"])
print("\n  (context) probe derivation_rule distribution - probe-level, not carrier-level:")
for r, n in drules.most_common():
    print(f"    {r:24s} {n}")

h263_pass = (f101 <= 0.40) and (f63 <= 0.40)
h263_verdict = "CONFIRMED" if h263_pass else "REFUTED (carriers spread across classes - no scoping leverage)"
RESULTS["H263"] = {
    "classification_used": "H230 name-shape (code-bearing/product-form/generic-concept/fragment); probe attribute/derivation_rule classify probes not carriers",
    "carriers_101": {"composition": comp101, "classes_covering_90pct": k101, "class_fraction": round(f101, 3)},
    "carriers_63": {"composition": comp63, "classes_covering_90pct": k63, "class_fraction": round(f63, 3)},
    "bar_class_fraction": 0.40, "pass": bool(h263_pass),
    "verdict_recommendation": h263_verdict,
}
print("H263 verdict:", h263_verdict)
log(f"H263 f101={f101:.2f} f63={f63:.2f} -> {h263_verdict}")

H263 101 probed carriers (n=101):
    code-bearing       11   10.9%
    product-form       85   84.2%
    generic-concept     3    3.0%
    fragment            2    2.0%
    classes covering >=90% = 2/4 = 50%  (bar <= 40%)  -> FAIL
H263 63 coverable carriers (n=63):
    code-bearing        5    7.9%
    product-form       56   88.9%
    generic-concept     0    0.0%
    fragment            2    3.2%
    classes covering >=90% = 2/4 = 50%  (bar <= 40%)  -> FAIL

  (context) probe derivation_rule distribution - probe-level, not carrier-level:
    catalogue_code           62
    doc_feature_generic      59
    doc_feature_branded      37
    spec_table_cell          20
    doc_spec_proximity       16
    spec_sentence            12
    doc_consolidation_spec   7
    feature_statement        6
H263 verdict: REFUTED (carriers spread across classes - no scoping leverage)


## Conclusions and report

Aggregates the six adjudications into the machine-readable report with per-clause metrics and per-gate `verdict_recommendation`, grounded in the cell outputs above.

In [12]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
report = {
    "round": "R24", "tier": "FANOUT free-gate (b) - CPU/disk only, zero LLM/GPU/Neo4j",
    "generated_utc": stamp,
    "gold_carrier_definition": "token_set_ratio(name.lower(), product.lower()) >= 85; union-of-5 = probed products matched by any of 5 arm-A runs pooled over 10 docs = 63",
    "anchors": {"union5": U, "single_run_coverage": round(single_cov, 4),
                "n_products": len(PRODUCTS), "runs": len(RUNS), "docs": len(DOCS)},
    "gates": RESULTS,
    "verdict_recommendation": {h: RESULTS[h]["verdict_recommendation"] for h in RESULTS},
}
out = REPORTS_DIR / f"fanout-gates-r24b-{stamp}.json"
out.write_text(json.dumps(report, indent=2))
log(f"report written: {out}")
log("=== r24b fanout gates done ===")

tbl = Table(box=box.SIMPLE, show_edge=False)
tbl.add_column("gate"); tbl.add_column("verdict_recommendation")
for h in RESULTS:
    tbl.add_row(h, RESULTS[h]["verdict_recommendation"])
console.print(tbl)
print("report written:", out)

 gate   verdict_recommendation                                                                      
────────────────────────────────────────────────────────────────────────────────────────────────────
 H255   REFUTED                                                                                     
 H256   REFUTED                                                                                     
 H258   CONFIRMED (cross-doc proxy)                                                                 
 H259   REFUTED (shadowing signal absent)                                                           
 H261   SELECTION: use the 3 low-leakage docs as held-out demos for the H261 LLM distillation probe 
 H263   REFUTED (carriers spread across classes - no scoping leverage)

report written: reports/fanout-gates-r24b-20260708T065520Z.json
